# トーラスのガウス曲率：3Dで回して、$b/a$ を動かす

[クリストッフェル記号とリーマン曲率テンソルのノート](../../../研究ノート/02_微分幾何/christoffel_riemann_intro.md#B-2)（付録B-2）のトーラスを、3Dで見ます。

**設定**：管の半径 $a=1$、中心軸から管の中心線までの距離 $b$（$b>a$）とし、

$$
\mathbf r(\theta,\phi)=\big((b+a\cos\theta)\cos\phi,\ (b+a\cos\theta)\sin\phi,\ a\sin\theta\big)
$$

です。ガウス曲率は、$\rho:=b+a\cos\theta$ として $K=\dfrac{\cos\theta}{a\rho}$ でした。外側（$\cos\theta>0$）は $K>0$（赤）、内側は $K<0$（青）です。

**使い方**：下部のスライダーで $b/a$ を変えます。3D の図は、ドラッグで回転、ホイールで拡大できます。

In [ ]:
import numpy as np
import sympy as sp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

a = 1.0

## 公式を、第一・第二基本形式から検算する

ノートの結果 $K=\cos\theta/(a\rho)$ を、SymPy で $K=\det L/\det g$ から計算し直して、一致することを確かめます。

In [ ]:
th, ph = sp.symbols("theta phi", real=True)
sa, sb = sp.symbols("a b", positive=True)
rho = sb + sa * sp.cos(th)
r = sp.Matrix([rho * sp.cos(ph), rho * sp.sin(ph), sa * sp.sin(th)])
r_th, r_ph = r.diff(th), r.diff(ph)
n = r_th.cross(r_ph)
n = sp.simplify(n / sp.sqrt(sp.simplify(n.dot(n))))
g = sp.Matrix([[r_th.dot(r_th), r_th.dot(r_ph)], [r_ph.dot(r_th), r_ph.dot(r_ph)]]).applyfunc(sp.simplify)
L = sp.Matrix([[r.diff(th, 2).dot(n), r.diff(th, ph).dot(n)], [r.diff(th, ph).dot(n), r.diff(ph, 2).dot(n)]]).applyfunc(sp.simplify)
K_sym = sp.simplify(L.det() / g.det())
print("K = det L / det g =", K_sym)
assert sp.simplify(K_sym - sp.cos(th) / (sa * (sb + sa * sp.cos(th)))) == 0
print("ノートの結果 cosθ/(aρ) と一致しました")

## ガウス・ボネの定理の確認

トーラスのオイラー標数は 0 なので、$\iint K\,dA=0$ のはずです。面積要素は $\sqrt{\det g}=a\rho$ なので、$K\sqrt{\det g}=\cos\theta$ となり、$\theta$ で積分すると 0 になります。$b/a$ の値に依らず、数値でも確かめます。

In [ ]:
def torus_data(b, nth=48, nph=96):
    th = np.linspace(-np.pi, np.pi, nth)
    ph = np.linspace(0, 2 * np.pi, nph)
    TH, PH = np.meshgrid(th, ph, indexing="ij")
    rho = b + a * np.cos(TH)
    return rho * np.cos(PH), rho * np.sin(PH), a * np.sin(TH), np.cos(TH) / (a * rho)


for b in [1.2, 2.4, 4.0]:
    th_f = np.linspace(-np.pi, np.pi, 4001)
    rho_f = b + a * np.cos(th_f)
    K_f = np.cos(th_f) / (a * rho_f)
    integral = 2 * np.pi * np.trapezoid(K_f * a * rho_f, th_f)   # ∬ K dA = ∫∫ K a ρ dθ dφ
    print(f"b/a = {b / a:.1f}:  ∬K dA = {integral:+.2e}")
    assert abs(integral) < 1e-9

## スライダーで動かす

左が3Dの図（色が $K$、赤が正・青が負）、右が $K$ と $K\sqrt{\det g}=\cos\theta$ のグラフです。

- $b/a$ を小さくすると、内側（$\theta=\pm180°$）の負の曲率が急に強くなります。
- ただし、面積で重みを付けた $K\sqrt{\det g}=\cos\theta$ は、$b/a$ に依らないので、正と負の面積はいつも打ち消し合います。

In [ ]:
bs = np.round(np.arange(1.2, 4.01, 0.1), 2)
th_deg = np.degrees(np.linspace(-np.pi, np.pi, 48))


def torus_traces(b):
    X, Y, Z, K = torus_data(b)
    surf = go.Surface(x=X, y=Y, z=Z, surfacecolor=K, colorscale="RdBu_r", cmid=0,
                      colorbar=dict(title="K", x=0.45, len=0.7))
    line = go.Scatter(x=th_deg, y=K[:, 0], mode="lines", line=dict(color="#d62728", width=3), name="K(θ)")
    return surf, line


frames, steps = [], []
for b in bs:
    surf, line = torus_traces(b)
    name = f"{b:.1f}"
    R = b + a + 0.3
    Kmin, Kmax = -1 / (a * (b - a)), 1 / (a * (b + a))
    frames.append(go.Frame(
        data=[surf, line], traces=[0, 1], name=name,
        layout=go.Layout(title_text=f"b/a = {b / a:.1f}　内側の最小値 K = {Kmin:.2f}",
                         scene=dict(xaxis=dict(range=[-R, R]), yaxis=dict(range=[-R, R]), zaxis=dict(range=[-R, R])),
                         yaxis2=dict(range=[min(-1.15, 1.1 * Kmin), 1.15]))))
    steps.append(dict(method="animate", label=name,
                      args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))

k0 = int(np.argmin(np.abs(bs - 2.4)))
surf0, line0 = torus_traces(bs[k0])
R0 = bs[k0] + a + 0.3
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]], column_widths=[0.6, 0.4])
fig.add_trace(surf0, row=1, col=1)
fig.add_trace(line0, row=1, col=2)
fig.add_trace(go.Scatter(x=th_deg, y=np.cos(np.radians(th_deg)), mode="lines",
                         line=dict(color="#444", dash="dash"), name="K√det g = cosθ"), row=1, col=2)
fig.frames = frames
fig.update_xaxes(title_text="θ [度]", range=[-180, 180], dtick=90, row=1, col=2)
fig.update_yaxes(title_text="K", range=[min(-1.15, -1.1 / (a * (bs[k0] - a))), 1.15], row=1, col=2)
fig.update_layout(
    title_text=frames[k0].layout.title.text,
    sliders=[dict(active=k0, currentvalue=dict(prefix="b/a = "), pad=dict(t=30), steps=steps)],
    scene=dict(aspectmode="cube", xaxis=dict(visible=False, range=[-R0, R0]), yaxis=dict(visible=False, range=[-R0, R0]),
               zaxis=dict(visible=False, range=[-R0, R0]), camera=dict(eye=dict(x=1.1, y=0.9, z=0.8))),
    height=650, margin=dict(l=0, r=0, t=60, b=0), legend=dict(orientation="h", x=0.58, y=1.07),
)
fig.show()

## 観察のポイント

- 赤（$K>0$）は外側、青（$K<0$）は内側です。上端と下端の円周（$\theta=\pm90°$）は $K=0$ で、白く見えます。
- 内側の最小値は $K=-1/(a(b-a))$ で、$b\to a$ で発散します。
- 右のグラフの破線（$K\sqrt{\det g}=\cos\theta$）は、スライダーを動かしても変わりません。正の部分と負の部分の面積が等しいので、$\iint K\,dA=0=2\pi\chi$（$\chi=0$）が成り立ちます。